# E6 — variantes de la cabeza de equipo neuronal (Colab, solo banco DFL)

Mismo procedimiento que E4 (`e4_owner_nn.ipynb`): 7 folds dejando un partido fuera + modelo final, 3 semillas. Tres variantes **fijadas antes de entrenar**, en este orden:

- **e6a**: pérdida auxiliar de cambio de equipo (H↔A directo a ±2 pasos = ±0,4 s), peso 0,5. La cabeza de cambio no se decodifica: solo regulariza.
- **e6b**: más capacidad y contexto: d = hidden = 128, 2 capas de atención entre jugadores, ventana de 20 s.
- **e6c**: e6a + e6b.

**Reglas:** aquí entra **solo** `colab_bundle_e6.zip` (banco DFL + código). Nada del reto. **Gemini desactivado.**

1. GPU A100 si es posible. 2. Sube `external/nn/colab_bundle_e6.zip` a `MyDrive/e6/`. 3. Celdas en orden; cada variante es reanudable. Se pueden lanzar en sesiones paralelas (celdas 1–2 y luego una celda de variante por sesión).
4. Descarga `MyDrive/e6/out_<variante>/` (oof_rnn.npz, owner_nn_s*.pt, owner_nn.json) y `log_<variante>.txt` a `external/nn/out_<variante>/` en local.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/e6'
import os; os.makedirs(WORK, exist_ok=True); print(os.listdir(WORK))

In [ ]:
import os, shutil
if not os.path.exists(f'{WORK}/colab_bundle_e6.zip'):
    from google.colab import files
    up = files.upload()
    shutil.move(next(iter(up)), f'{WORK}/colab_bundle_e6.zip')
!rm -rf /content/e6 && mkdir -p /content/e6 && unzip -q {WORK}/colab_bundle_e6.zip -d /content/e6 && ls -la /content/e6 /content/e6/src

In [ ]:
# Smoke (~2-5 min): simetría exacta + un fold diminuto con la variante más grande
%cd /content/e6
cfg = '{"aux": 0.5, "d": 128, "hidden": 128, "layers": 2, "win": 100, "hop": 50}'
!python -m pytest -q tests/test_nn_inputs.py
!python -m src.owner_nn --data nn_bench.npz --out /content/smoke --smoke --cfg '{cfg}'

In [ ]:
# e6a: 7 folds + final, 3 semillas. Reanudable.
%cd /content/e6
cfg = '{"aux": 0.5}'
!python -m src.owner_nn --data nn_bench.npz --out {WORK}/out_e6a --cfg '{cfg}' 2>&1 | tee -a {WORK}/log_e6a.txt

In [ ]:
# e6b: 7 folds + final, 3 semillas. Reanudable.
%cd /content/e6
cfg = '{"d": 128, "hidden": 128, "layers": 2, "win": 100, "hop": 50}'
!python -m src.owner_nn --data nn_bench.npz --out {WORK}/out_e6b --cfg '{cfg}' 2>&1 | tee -a {WORK}/log_e6b.txt

In [ ]:
# e6c: 7 folds + final, 3 semillas. Reanudable.
%cd /content/e6
cfg = '{"aux": 0.5, "d": 128, "hidden": 128, "layers": 2, "win": 100, "hop": 50}'
!python -m src.owner_nn --data nn_bench.npz --out {WORK}/out_e6c --cfg '{cfg}' 2>&1 | tee -a {WORK}/log_e6c.txt